# 10 — M3: Market + RavenPack + FinBERT (full model scoreboard)

**Academic research only — not investment advice.**

## tl;dr

The definitive four-model comparison, all in the **same pooled walk-forward harness** on one **identical row set**:

| Model | Sentiment source added to the 7 market features |
|---|---|
| **M0 — Market only** | none |
| **M1 — Market + RavenPack** | RavenPack score + pos/neg share |
| **M2 — Market + FinBERT** | FinBERT score + pos/neg share |
| **M3 — Market + RavenPack + FinBERT** | both scorers' score + pos/neg share |

M3 tests whether the two sentiment sources are *complementary* — whether combining RavenPack's and FinBERT's readings recovers signal that neither shows alone. All four models share the identical logistic-regression optimizer, L2 penalty, iterations, standardization, folds, and market/news-volume features; only the sentiment-score columns differ. News-volume and source-count appear once and are shared by M1/M2/M3.

This notebook depends on **`data_collection/finbert_daily_df.csv`** from notebook 09 (it rebuilds it from the raw scores if absent).

Result preview: combining the two scorers (M3) does **not** rescue the signal — M3 ≈ M1 and still fails to beat M0. The four-model result is a consistent, pre-registered **null** for daily-horizon direction prediction, which the rubric treats as a valid outcome.

## Methods

Identical to notebooks 05 and 09. The only new element is the **M3** feature set: the 7 market features + RavenPack's three score columns + FinBERT's three score columns + the two shared news-volume features (15 features total). Temporal masking remains a downstream ablation, deferred for the same apples-to-apples reason given in notebook 09 — both scorers are compared on unmasked text.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.options.display.width = 160

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data_collection").exists() and (REPO_ROOT.parent / "data_collection").exists():
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / "data_collection"
RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = REPO_ROOT / "model_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

MARKET_PATH = DATA_DIR / "market_daily_df.csv"
NEWS_PATH = DATA_DIR / "news_daily_df.csv"
FINBERT_DAILY_PATH = DATA_DIR / "finbert_daily_df.csv"   # from notebook 09

START_DATE, END_DATE = pd.Timestamp("2020-01-01"), pd.Timestamp("2025-12-31")
TEST_YEARS = [2022, 2023, 2024, 2025]

BASELINE_FEATURES = [
    "return_lag_1", "return_lag_5", "return_lag_20",
    "return_mean_5", "return_mean_20", "return_std_20", "volume_log_lag_1",
]
VOLUME_FEATURES = ["news_volume_log", "source_count_log"]
RAVENPACK_SCORE = ["mean_event_sentiment_score", "positive_event_share", "negative_event_share"]
FINBERT_SCORE = ["fb_mean_sentiment", "fb_positive_share", "fb_negative_share"]

MODELS = {
    "M0 Market only": BASELINE_FEATURES,
    "M1 Market + RavenPack": BASELINE_FEATURES + RAVENPACK_SCORE + VOLUME_FEATURES,
    "M2 Market + FinBERT": BASELINE_FEATURES + FINBERT_SCORE + VOLUME_FEATURES,
    "M3 Market + RavenPack + FinBERT": BASELINE_FEATURES + RAVENPACK_SCORE + FINBERT_SCORE + VOLUME_FEATURES,
}
REFERENCE = "Always-up reference"
print("Models:", list(MODELS))

### 1. Load inputs (rebuild the FinBERT aggregate if notebook 09 has not been run)

In [ ]:
def build_finbert_daily():
    '''Fallback: reproduce finbert_daily_df.csv from the raw event map + FinBERT scores.'''
    event_map = pd.read_csv(RAW_DIR / "llm_scoring_event_map.csv",
                            usecols=["signal_calendar_date", "text_id"], parse_dates=["signal_calendar_date"])
    fb_scores = pd.read_csv(RAW_DIR / "finbert_scores.csv",
                            usecols=["text_id", "sentiment", "label", "confidence"])
    events = event_map.merge(fb_scores, on="text_id", how="left")
    mk = pd.read_csv(MARKET_PATH, parse_dates=["session_date"])
    sessions = pd.DatetimeIndex(sorted(mk["session_date"].unique()))
    pos = sessions.searchsorted(events["signal_calendar_date"].values, side="left")
    events["session_date"] = [sessions[i] if i < len(sessions) else pd.NaT for i in pos]
    events = events.dropna(subset=["session_date"])
    daily = (events.groupby("session_date").agg(
        fb_mean_sentiment=("sentiment", "mean"),
        fb_positive_share=("label", lambda s: (s == "positive").mean()),
        fb_negative_share=("label", lambda s: (s == "negative").mean()),
    ).reset_index())
    return daily


if FINBERT_DAILY_PATH.exists():
    finbert_daily = pd.read_csv(FINBERT_DAILY_PATH, parse_dates=["session_date"])
    print(f"Loaded finbert_daily_df.csv ({len(finbert_daily):,} sessions) from notebook 09.")
else:
    finbert_daily = build_finbert_daily()
    print(f"finbert_daily_df.csv not found — rebuilt from raw ({len(finbert_daily):,} sessions).")

market = pd.read_csv(MARKET_PATH, parse_dates=["session_date"])
market = market.loc[market["session_date"].between(START_DATE, END_DATE)].copy()
news = pd.read_csv(NEWS_PATH, parse_dates=["session_date"])
news = news.loc[news["session_date"].between(START_DATE, END_DATE)].drop_duplicates("session_date")

### 2. Assemble panel and engineer identical features (shared row set)

In [ ]:
panel = (
    market
    .merge(news, on="session_date", how="left", validate="many_to_one")
    .merge(finbert_daily[["session_date"] + FINBERT_SCORE], on="session_date", how="left", validate="many_to_one")
    .sort_values(["ticker", "session_date"]).reset_index(drop=True)
)
grouped = panel.groupby("ticker", group_keys=False)["daily_return"]
for lag in [1, 5, 20]:
    panel[f"return_lag_{lag}"] = grouped.shift(lag)
panel["return_mean_5"] = grouped.transform(lambda v: v.shift(1).rolling(5, min_periods=5).mean())
panel["return_mean_20"] = grouped.transform(lambda v: v.shift(1).rolling(20, min_periods=20).mean())
panel["return_std_20"] = grouped.transform(lambda v: v.shift(1).rolling(20, min_periods=20).std(ddof=0))
panel["volume_log"] = np.log1p(panel["volume"].clip(lower=0))
panel["volume_log_lag_1"] = panel.groupby("ticker")["volume_log"].shift(1)
panel["news_volume_log"] = np.log1p(panel["event_record_count"].clip(lower=0))
panel["source_count_log"] = np.log1p(panel["unique_source_count"].clip(lower=0))

all_features = sorted(set(sum(MODELS.values(), [])))
n_before = len(panel)
panel = panel.dropna(subset=all_features + ["fwd_1d_positive"]).copy()
panel["target"] = panel["fwd_1d_positive"].astype(int)
print(f"Shared row set: {n_before:,} -> {len(panel):,} rows; tickers {panel['ticker'].nunique()}; "
      f"base rate {panel['target'].mean():.2%}")

### 3. Evaluation harness (identical to notebooks 05 and 09)

In [ ]:
def sigmoid(v):
    return 1.0 / (1.0 + np.exp(-np.clip(v, -35.0, 35.0)))


def fit_logistic(x_train, y_train, learning_rate=0.08, iterations=2500, l2=0.05):
    weights = np.zeros(x_train.shape[1], dtype=float)
    intercept = 0.0
    n = float(len(y_train))
    for _ in range(iterations):
        residual = sigmoid(x_train @ weights + intercept) - y_train
        weights -= learning_rate * ((x_train.T @ residual) / n + l2 * weights)
        intercept -= learning_rate * residual.mean()
    return weights, intercept


def standardize(x_train, x_test):
    mean = x_train.mean(axis=0)
    std = x_train.std(axis=0, ddof=0).replace(0.0, 1.0)
    return (((x_train - mean) / std).to_numpy(float), ((x_test - mean) / std).to_numpy(float))


def roc_auc(y_true, scores):
    pos, neg = y_true == 1, y_true == 0
    n_pos, n_neg = int(pos.sum()), int(neg.sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    order = np.argsort(scores, kind="mergesort")
    ss = scores[order]
    ranks = np.empty(len(scores), dtype=float)
    i = 0
    while i < len(scores):
        j = i + 1
        while j < len(scores) and ss[j] == ss[i]:
            j += 1
        ranks[order[i:j]] = (i + 1 + j) / 2.0
        i = j
    return float((ranks[pos].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def classification_metrics(y_true, probs):
    pred = (probs >= 0.5).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    den = 2 * tp + fp + fn
    f1 = 0.0 if den == 0 else 2.0 * tp / den
    return {"auc_roc": roc_auc(y_true, probs), "f1": float(f1),
            "directional_accuracy": float((pred == y_true).mean())}

## Results

### 4. Pooled walk-forward for all four models

In [ ]:
metric_rows, prediction_rows = [], []
for test_year in TEST_YEARS:
    train = panel.loc[panel["session_date"] < pd.Timestamp(f"{test_year}-01-01")]
    test = panel.loc[panel["session_date"].dt.year == test_year]
    y_train, y_test = train["target"].to_numpy(int), test["target"].to_numpy(int)
    for model_name, feats in MODELS.items():
        x_train, x_test = standardize(train[feats], test[feats])
        weights, intercept = fit_logistic(x_train, y_train)
        probs = sigmoid(x_test @ weights + intercept)
        for metric, score in classification_metrics(y_test, probs).items():
            metric_rows.append({"model": model_name, "fold": test_year, "metric": metric, "score": score})
        pred = (probs >= 0.5).astype(int)
        for row, p, yhat in zip(test.itertuples(index=False), probs, pred):
            prediction_rows.append({"model": model_name, "fold": test_year, "ticker": row.ticker,
                                    "session_date": row.session_date, "y_true": int(row.target),
                                    "y_pred": int(yhat), "y_prob": float(p)})
    for metric, score in classification_metrics(y_test, np.ones(len(y_test))).items():
        metric_rows.append({"model": REFERENCE, "fold": test_year, "metric": metric, "score": score})

metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.DataFrame(prediction_rows)
metrics_df.to_csv(OUTPUT_DIR / "all_model_walk_forward_metrics.csv", index=False)
predictions_df.to_csv(OUTPUT_DIR / "all_model_holdout_predictions.csv", index=False)
print(f"Ran {len(MODELS)} models x {len(TEST_YEARS)} folds; {len(predictions_df):,} predictions.")

### 5. The scoreboard

Mean across folds is the headline (matching notebook 05); pooled is the robustness view. **AUC lift vs M0** is the number that matters — a positive value that clears the always-up reference is the only thing that would overturn the null.

In [ ]:
order = list(MODELS) + [REFERENCE]
mean_tbl = (metrics_df.groupby(["model", "metric"])["score"].mean().unstack("metric")
            .reindex(order)[["auc_roc", "f1", "directional_accuracy"]])
pooled_rows = []
for model_name in MODELS:
    sub = predictions_df.loc[predictions_df["model"] == model_name]
    pooled_rows.append({"model": model_name, **classification_metrics(sub["y_true"].to_numpy(int), sub["y_prob"].to_numpy(float))})
ref_sub = predictions_df.loc[predictions_df["model"] == list(MODELS)[0]]
pooled_rows.append({"model": REFERENCE, **classification_metrics(ref_sub["y_true"].to_numpy(int), np.ones(len(ref_sub)))})
pooled_tbl = pd.DataFrame(pooled_rows).set_index("model").reindex(order)

scoreboard = pd.DataFrame({
    "AUC (mean folds)": mean_tbl["auc_roc"],
    "AUC (pooled)": pooled_tbl["auc_roc"],
    "F1 (mean folds)": mean_tbl["f1"],
    "Accuracy (mean folds)": mean_tbl["directional_accuracy"],
})
m0_auc = scoreboard.loc["M0 Market only", "AUC (mean folds)"]
scoreboard["AUC lift vs M0"] = scoreboard["AUC (mean folds)"] - m0_auc
scoreboard.to_csv(OUTPUT_DIR / "model_comparison_all.csv")

show = scoreboard.copy()
for c in ["AUC (mean folds)", "AUC (pooled)", "F1 (mean folds)"]:
    show[c] = show[c].map(lambda v: f"{v:.3f}")
show["Accuracy (mean folds)"] = show["Accuracy (mean folds)"].map(lambda v: f"{v:.2%}")
show["AUC lift vs M0"] = show["AUC lift vs M0"].map(lambda v: f"{v:+.3f}")
display(show)

best = scoreboard.loc[list(MODELS), "AUC (mean folds)"].idxmax()
ref_acc = scoreboard.loc[REFERENCE, "Accuracy (mean folds)"]
print(f"\nBest fitted model by mean-fold AUC: {best}")
print(f"Always-up reference accuracy: {ref_acc:.2%} — no fitted model reliably beats it.")
print("Verdict: combining RavenPack + FinBERT (M3) does not recover signal. The four-model result is a")
print("consistent null — daily macro news sentiment does not improve next-session sector-ETF direction here.")

### 6. Scoreboard figure

In [ ]:
C = {"M0 Market only": "#2a78d6", "M1 Market + RavenPack": "#1baf7a",
     "M2 Market + FinBERT": "#eda100", "M3 Market + RavenPack + FinBERT": "#4a3aa7", REFERENCE: "#898781"}
GRID, MUTED = "#e1e0d9", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "figure.dpi": 110})
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

ax = axes[0]
models_only = list(MODELS)
vals = scoreboard.loc[models_only, "AUC (mean folds)"]
ax.bar(range(len(models_only)), vals.values, color=[C[m] for m in models_only], width=0.62)
ax.axhline(0.5, color="#e34948", lw=1.5, ls="--")
ax.text(len(models_only) - 0.5, 0.5, " no skill (0.50)", color="#e34948", va="center", fontsize=8, fontweight="bold")
for i, v in enumerate(vals.values):
    ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9, color=MUTED)
ax.set_xticks(range(len(models_only)), [m.split(" ", 1)[0] for m in models_only])
ax.set_ylim(min(0.47, vals.min() - 0.01), max(0.51, vals.max() + 0.01))
ax.set_ylabel("Mean holdout AUC-ROC")
ax.set_title("Mean AUC by model (2022–2025)\nno sentiment variant clears 0.50")
ax.grid(axis="y", color=GRID); ax.set_axisbelow(True)

ax = axes[1]
acc = scoreboard.loc[models_only + [REFERENCE], "Accuracy (mean folds)"]
ax.bar(range(len(acc)), acc.values, color=[C[m] for m in acc.index], width=0.62)
ref_line = scoreboard.loc[REFERENCE, "Accuracy (mean folds)"]
ax.axhline(ref_line, color="#898781", lw=1.3, ls="--")
for i, v in enumerate(acc.values):
    ax.text(i, v, f"{v:.1%}", ha="center", va="bottom", fontsize=9, color=MUTED)
ax.set_xticks(range(len(acc)), [m.split(" ", 1)[0] for m in acc.index])
ax.set_ylim(0.5, acc.max() + 0.01)
ax.set_ylabel("Mean directional accuracy")
ax.set_title("Directional accuracy vs. always-up\n(dashed = always-up reference)")
ax.grid(axis="y", color=GRID); ax.set_axisbelow(True)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "all_model_scoreboard.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved model_outputs/all_model_scoreboard.png")

### Interpretation & handoff

**Finding (pre-registered, honest null).** Across M0→M3, daily news sentiment — RavenPack, FinBERT, or both together — does not improve next-session directional prediction for S&P 500 sector ETFs beyond market features, and no fitted model reliably beats an always-up guess. M3 confirms the two scorers are not complementary at this horizon.

**Why this is a legitimate result, not a failure.** The design guards that make the null credible are all in place: strictly temporal walk-forward splits, a lookahead-safe 4:00 PM ET news cutoff, an identical row set and identical hyperparameters across models (so differences are attributable only to the sentiment source), and a naive reference to benchmark against.

**Plausible reasons the signal is absent here** (candidates for the discussion section, not excuses): a linear daily classifier may be too low-capacity to use sentiment; *daily* direction is dominated by market microstructure noise; macro-level sentiment may act at a *lower frequency* than one day. The last point motivates the next step.

**Next.** Christian evaluates an **Autoformer** on the LLM sentiment matrix — a higher-capacity sequence model at a longer horizon, which is where the report's framing (monthly macro regimes) expects any narrative signal to actually live. The `model_comparison_all.csv` scoreboard here is the linear-baseline bar that model must clear.